# FEC House Campaign Finance Data 2023-2024

## Data Cleaning & Preparation

Source: Federal Election Commission (https://www.fec.gov/data/)
File: webl24.txt

**Steps performed:**
1. Loaded pipe separated txt file with manually assigned column names
2. Filtered for House candidates only using CAND_ID prefix 'H'
3. Removed Presidential candidates (Biden, Harris) found in the file
4. Identified and removed 3 House members misclassified under House IDs 
   who were actually running for Senate in 2024 (Gallego, Schiff, Trone)
5. Dropped columns entirely null (election result columns only available in 1996-2006 files)
6. Standardized party values to REPUBLICAN, DEMOCRAT, OTHER
7. Fixed null district value for Delaware (at-large state)
8. Reformatted district column to match Harvard data format (0, 1, 2, 3)

**Output:** fec_house_2024_clean.csv

In [1]:
import pandas as pd
import os

columns = [
    'CAND_ID', 'CAND_NAME', 'CAND_ICI', 'PTY_CD', 'CAND_PTY_AFFILIATION',
    'TTL_RECEIPTS', 'TRANS_FROM_AUTH', 'TTL_DISB', 'TRANS_TO_AUTH',
    'COH_BOP', 'COH_COP', 'CAND_CONTRIB', 'CAND_LOANS', 'OTHER_LOANS',
    'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY', 'DEBTS_OWED_BY', 'TTL_INDIV_CONTRIB',
    'CAND_OFFICE_ST', 'CAND_OFFICE_DISTRICT', 'SPEC_ELECTION', 'PRIM_ELECTION',
    'RUN_ELECTION', 'GEN_ELECTION', 'GEN_ELECTION_PRECENT', 'OTHER_POL_CMTE_CONTRIB',
    'POL_PTY_CONTRIB', 'CVG_END_DT', 'INDIV_REFUNDS', 'CMTE_REFUNDS'
]

In [2]:
fec_df = pd.read_csv('j_notebooks/raw_source/webl24.txt', sep='|', names=columns, header=None)

fec_df.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS
0,H2AK01158,"PELTOLA, MARY",I,1,DEM,13443537.46,951851.88,14050828.27,0.00,691260.30,...,NaN,NaN,NaN,NaN,NaN,1615986.30,9969.28,12/31/2024,161309.36,5625.0
1,H2AK01083,"BEGICH, NICHOLAS III",C,2,REP,2810467.65,176570.41,2747371.58,17659.66,41233.99,...,NaN,NaN,NaN,NaN,NaN,318750.00,5000.00,12/31/2024,23031.99,0.0
2,H4AK00156,"DAHLSTROM, NANCY",C,2,REP,996163.60,435712.04,790351.61,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,262916.02,0.00,12/31/2024,3218.30,0.0
3,H4AL01255,"HOLMES, THOMAS BETHUNE MR.",C,1,DEM,17698.86,0.00,16817.50,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,2002.63,0.00,12/31/2024,0.00,0.0
4,H0AL01055,"CARL, JERRY LEE, JR",I,2,REP,2246839.19,547807.76,2631446.59,27316.59,453897.82,...,NaN,NaN,NaN,NaN,NaN,634500.00,0.00,12/31/2024,193499.00,84500.0


In [3]:
fec_df.columns

Index(['CAND_ID', 'CAND_NAME', 'CAND_ICI', 'PTY_CD', 'CAND_PTY_AFFILIATION',
       'TTL_RECEIPTS', 'TRANS_FROM_AUTH', 'TTL_DISB', 'TRANS_TO_AUTH',
       'COH_BOP', 'COH_COP', 'CAND_CONTRIB', 'CAND_LOANS', 'OTHER_LOANS',
       'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY', 'DEBTS_OWED_BY',
       'TTL_INDIV_CONTRIB', 'CAND_OFFICE_ST', 'CAND_OFFICE_DISTRICT',
       'SPEC_ELECTION', 'PRIM_ELECTION', 'RUN_ELECTION', 'GEN_ELECTION',
       'GEN_ELECTION_PRECENT', 'OTHER_POL_CMTE_CONTRIB', 'POL_PTY_CONTRIB',
       'CVG_END_DT', 'INDIV_REFUNDS', 'CMTE_REFUNDS'],
      dtype='object')

In [4]:
fec_df.shape

(2378, 30)

In [5]:
# Checking for nulls
fec_df.isnull().sum()

CAND_ID                      0
CAND_NAME                    0
CAND_ICI                     5
PTY_CD                       0
CAND_PTY_AFFILIATION         1
TTL_RECEIPTS                 0
TRANS_FROM_AUTH              0
TTL_DISB                     0
TRANS_TO_AUTH                0
COH_BOP                      0
COH_COP                      0
CAND_CONTRIB                 0
CAND_LOANS                   0
OTHER_LOANS                  0
CAND_LOAN_REPAY              0
OTHER_LOAN_REPAY             0
DEBTS_OWED_BY                0
TTL_INDIV_CONTRIB            0
CAND_OFFICE_ST               0
CAND_OFFICE_DISTRICT        27
SPEC_ELECTION             2378
PRIM_ELECTION             2378
RUN_ELECTION              2378
GEN_ELECTION              2378
GEN_ELECTION_PRECENT      2378
OTHER_POL_CMTE_CONTRIB       0
POL_PTY_CONTRIB              0
CVG_END_DT                   0
INDIV_REFUNDS                0
CMTE_REFUNDS                 0
dtype: int64

In [6]:
# Checking party values
fec_df['CAND_PTY_AFFILIATION'].value_counts()

CAND_PTY_AFFILIATION
REP    1139
DEM     992
IND      78
LIB      37
OTH      18
GRE      16
UN       14
NPA      13
W        12
DFL       9
NNE       8
NON       6
NOP       6
CON       5
IDP       5
UNK       3
AMP       2
CRV       2
NPP       2
UNA       1
SOC       1
SEP       1
BDY       1
GOP       1
UUP       1
PNP       1
MO        1
UNI       1
IAP       1
Name: count, dtype: int64

In [7]:
# Basic stats on finance columns
fec_df[['TTL_RECEIPTS', 'TTL_DISB']].describe()

,TTL_RECEIPTS,TTL_DISB
count,2.378000e+03,2.378000e+03
mean,2.734584e+06,2.829207e+06
std,3.462681e+07,3.502467e+07
min,0.000000e+00,0.000000e+00
25%,1.831000e+04,1.677364e+04
50%,1.409575e+05,1.348703e+05
75%,1.216854e+06,1.160113e+06
max,1.175189e+09,1.175285e+09


In [8]:
# Checking outliers that raised over a billion dollars
fec_df[fec_df['TTL_RECEIPTS'] == fec_df['TTL_RECEIPTS'].max()][['CAND_NAME', 'TTL_RECEIPTS', 'TTL_DISB']]

,CAND_NAME,TTL_RECEIPTS,TTL_DISB
1931,"HARRIS, KAMALA",1.175189e+09,1.175285e+09
1955,"BIDEN, JOSEPH R JR",1.175189e+09,1.175215e+09


## Filtering & Data Quality Checks

Filtering for House candidates only using CAND_ID prefix 'H', since the file 
also contains Senate and Presidential candidates.

**Data quality issues found and resolved:**
- Removed Presidential candidates (Biden, Harris) who were included in the file
- Identified 3 House members who ran for Senate in 2024 and were still filed 
  under their House CAND_ID:
    - Ruben Gallego (AZ) - \$64.6M raised
    - Adam Schiff (CA) - \$48.1M raised
    - David Trone (MD) - \$63.8M raised
- Hakeem Jeffries (NY) - \$22.9M raised was **kept** as he was legitimately 
  running for his House seat as House Minority Leader

In [9]:
# Filtering for House Candidates Only
fec_df = fec_df[fec_df['CAND_ID'].str.startswith('H')]

fec_df.shape

(1927, 30)

In [10]:
fec_df[['TTL_RECEIPTS', 'TTL_DISB']].describe()

,TTL_RECEIPTS,TTL_DISB
count,1.927000e+03,1.927000e+03
mean,1.180744e+06,1.111112e+06
std,3.140795e+06,3.197249e+06
min,0.000000e+00,0.000000e+00
25%,2.426793e+04,2.180678e+04
50%,1.675172e+05,1.634129e+05
75%,1.238557e+06,1.159960e+06
max,6.465720e+07,6.572262e+07


In [11]:
# Checking an outlier that raised over $64 million for a house race
fec_df[fec_df['TTL_RECEIPTS'] == fec_df['TTL_RECEIPTS'].max()][['CAND_NAME', 'CAND_OFFICE_ST', 'TTL_RECEIPTS']]

,CAND_NAME,CAND_OFFICE_ST,TTL_RECEIPTS
62,"GALLEGO, RUBEN",AZ,64657199.96


In [12]:
fec_df[fec_df['CAND_NAME'] == 'GALLEGO, RUBEN'][['CAND_ID', 'CAND_NAME', 'CAND_OFFICE_ST']]

,CAND_ID,CAND_NAME,CAND_OFFICE_ST
62,H4AZ07043,"GALLEGO, RUBEN",AZ


In [13]:
# Checking for other potential Senate candidates in House data
fec_df[fec_df['TTL_RECEIPTS'] > 20000000][['CAND_ID', 'CAND_NAME', 'CAND_OFFICE_ST', 'TTL_RECEIPTS']]

,CAND_ID,CAND_NAME,CAND_OFFICE_ST,TTL_RECEIPTS
62,H4AZ07043,"GALLEGO, RUBEN",AZ,64657199.96
220,H0CA27085,"SCHIFF, ADAM",CA,48149196.78
844,H6MD08549,"TRONE, DAVID",MD,63833136.59
1238,H2NY10092,"JEFFRIES, HAKEEM",NY,22977197.34


In [14]:
senate_candidates = ['H4AZ07043', 'H0CA27085', 'H6MD08549']

fec_df = fec_df[~fec_df['CAND_ID'].isin(senate_candidates)]

fec_df.shape

(1924, 30)

## Cleaning FEC Data

**Steps performed:**
1. Drop columns that are entirely null (election result columns only available in 1996-2006 files)
2. Standardize party values to REPUBLICAN, DEMOCRAT, OTHER
3. Fix null district values for at-large states
4. Reformat district column to match Harvard data format (0, 1, 2, 3 etc)

In [15]:
# Dropping columns that are all null
fec_df = fec_df.drop(columns=[
    'SPEC_ELECTION', 'PRIM_ELECTION', 
    'RUN_ELECTION', 'GEN_ELECTION', 
    'GEN_ELECTION_PRECENT'
])

fec_df.shape

(1924, 25)

In [16]:
# Standardizing party values
fec_df['CAND_PTY_AFFILIATION'] = fec_df['CAND_PTY_AFFILIATION'].apply(
    lambda x: 'REPUBLICAN' if x == 'REP' else 
              'DEMOCRAT' if x in ['DEM', 'DFL'] else 
              'OTHER'
)

fec_df['CAND_PTY_AFFILIATION'].value_counts()

CAND_PTY_AFFILIATION
REPUBLICAN    947
DEMOCRAT      868
OTHER         109
Name: count, dtype: int64

In [17]:
# checking null districts
fec_df[fec_df['CAND_OFFICE_DISTRICT'].isnull()][['CAND_ID', 'CAND_NAME', 'CAND_OFFICE_ST', 'CAND_OFFICE_DISTRICT']]

,CAND_ID,CAND_NAME,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT
412,H4DE00078,"WHALEN III, JOHN J",DE,NaN


In [18]:
# fixing it
fec_df['CAND_OFFICE_DISTRICT'] = fec_df['CAND_OFFICE_DISTRICT'].fillna('00')

# Confirming no more nulls
fec_df['CAND_OFFICE_DISTRICT'].isnull().sum()

0

In [19]:
# Checking all states with district 00
fec_df[fec_df['CAND_OFFICE_DISTRICT'] == '00']['CAND_OFFICE_ST'].value_counts()

CAND_OFFICE_ST
DE    1
Name: count, dtype: int64

In [20]:
fec_df['CAND_OFFICE_DISTRICT'].value_counts().head(20)

CAND_OFFICE_DISTRICT
3.0     213
2.0     201
1.0     169
6.0     136
4.0     119
5.0     119
8.0      99
7.0      97
10.0     91
9.0      59
13.0     49
0.0      46
11.0     44
12.0     41
14.0     35
16.0     32
30.0     21
15.0     21
26.0     21
32.0     21
Name: count, dtype: int64

In [21]:
fec_df[fec_df['CAND_OFFICE_ST'] == 'AK'][['CAND_ID', 'CAND_NAME', 'CAND_OFFICE_ST', 'CAND_OFFICE_DISTRICT']]

,CAND_ID,CAND_NAME,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT
0,H2AK01158,"PELTOLA, MARY",AK,0.0
1,H2AK01083,"BEGICH, NICHOLAS III",AK,0.0
2,H4AK00156,"DAHLSTROM, NANCY",AK,0.0


In [22]:
# Converting district to integer first to remove decimals then to string
fec_df['CAND_OFFICE_DISTRICT'] = fec_df['CAND_OFFICE_DISTRICT'].astype(float).astype(int).astype(str)

# Pad with leading zero so 1 becomes 01, 2 becomes 02 etc
fec_df['CAND_OFFICE_DISTRICT'] = fec_df['CAND_OFFICE_DISTRICT'].str.zfill(2)

# Check it
fec_df['CAND_OFFICE_DISTRICT'].value_counts().head(10)

CAND_OFFICE_DISTRICT
03    213
02    201
01    169
06    136
04    119
05    119
08     99
07     97
10     91
09     59
Name: count, dtype: int64

In [23]:
# Removing leading zeros to match Harvard format
fec_df['CAND_OFFICE_DISTRICT'] = fec_df['CAND_OFFICE_DISTRICT'].astype(int).astype(str)

fec_df['CAND_OFFICE_DISTRICT'].value_counts().head(10)

CAND_OFFICE_DISTRICT
3     213
2     201
1     169
6     136
4     119
5     119
8      99
7      97
10     91
9      59
Name: count, dtype: int64

## Final Check

In [24]:
# Checking for any remaining nulls
fec_df.isnull().sum()

CAND_ID                   0
CAND_NAME                 0
CAND_ICI                  0
PTY_CD                    0
CAND_PTY_AFFILIATION      0
TTL_RECEIPTS              0
TRANS_FROM_AUTH           0
TTL_DISB                  0
TRANS_TO_AUTH             0
COH_BOP                   0
COH_COP                   0
CAND_CONTRIB              0
CAND_LOANS                0
OTHER_LOANS               0
CAND_LOAN_REPAY           0
OTHER_LOAN_REPAY          0
DEBTS_OWED_BY             0
TTL_INDIV_CONTRIB         0
CAND_OFFICE_ST            0
CAND_OFFICE_DISTRICT      0
OTHER_POL_CMTE_CONTRIB    0
POL_PTY_CONTRIB           0
CVG_END_DT                0
INDIV_REFUNDS             0
CMTE_REFUNDS              0
dtype: int64

In [25]:
# Quick look at the clean data
fec_df.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,OTHER_LOAN_REPAY,DEBTS_OWED_BY,TTL_INDIV_CONTRIB,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS
0,H2AK01158,"PELTOLA, MARY",I,1,DEMOCRAT,13443537.46,951851.88,14050828.27,0.00,691260.30,...,0.0,0.00,10800004.13,AK,0,1615986.30,9969.28,12/31/2024,161309.36,5625.0
1,H2AK01083,"BEGICH, NICHOLAS III",C,2,REPUBLICAN,2810467.65,176570.41,2747371.58,17659.66,41233.99,...,0.0,425000.00,2309088.23,AK,0,318750.00,5000.00,12/31/2024,23031.99,0.0
2,H4AK00156,"DAHLSTROM, NANCY",C,2,REPUBLICAN,996163.60,435712.04,790351.61,0.00,0.00,...,0.0,0.00,286828.61,AK,0,262916.02,0.00,12/31/2024,3218.30,0.0
3,H4AL01255,"HOLMES, THOMAS BETHUNE MR.",C,1,DEMOCRAT,17698.86,0.00,16817.50,0.00,0.00,...,0.0,0.00,3076.82,AL,1,2002.63,0.00,12/31/2024,0.00,0.0
4,H0AL01055,"CARL, JERRY LEE, JR",I,2,REPUBLICAN,2246839.19,547807.76,2631446.59,27316.59,453897.82,...,0.0,254535.87,1063039.38,AL,1,634500.00,0.00,12/31/2024,193499.00,84500.0


In [26]:
# Final shape
fec_df.shape

(1924, 25)

In [27]:
fec_df.to_csv('j_notebooks/clean_data/fec_house_2024_clean.csv', index=False)

print('File exported!')

File exported!
